In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [ ]:
## Alright, let's see what we got here...
df = pd.read_csv('cbb.csv')
df["POSTSEASON"].value_counts()

POSTSEASON
R64          352
R32          176
S16           88
E8            44
R68           44
F4            22
2ND           11
Champions     11
Name: count, dtype: int64

In [ ]:
##Adding a numerical classification to the POSTSEASON column to make it easier to analyze and visualize the data. Honestly, ape brain. Big number = good. Small number = bad.
df["Post_Season_Score"] = np.select(
    [df["POSTSEASON"] == "Champions", df["POSTSEASON"] == "2ND", df["POSTSEASON"] == "F4", 
     df["POSTSEASON"] == "E8", df["POSTSEASON"] == "S16", df["POSTSEASON"] == "R32", 
     df["POSTSEASON"] == "R64"],
    [7, 6, 5, 4, 3, 2, 1],
    default=0
)

In [4]:
##Changing Seed to Reflect "Power"
df["Seed_Power"] = 17 - df["SEED"]
df.columns


Index(['TEAM', 'CONF', 'G', 'W', 'ADJOE', 'ADJDE', 'BARTHAG', 'EFG_O', 'EFG_D',
       'TOR', 'TORD', 'ORB', 'DRB', 'FTR', 'FTRD', '2P_O', '2P_D', '3P_O',
       '3P_D', 'ADJ_T', 'WAB', 'POSTSEASON', 'SEED', 'YEAR',
       'Post_Season_Score', 'Seed_Power'],
      dtype='object')

In [ ]:
## Renaming columns for Clarity. Imma need a cheat sheat. Seriously, I thought Field Goal was football, not basketball.
df.rename(columns={"G": "Games Played", 
                   "W": "Wins",
                   "ADJOE": "Adjusted Offensive Efficiency",
                   "ADJDE": "Adjusted Defensive Efficiency",
                   "BARTHAG": "Bartag Score",
                   "EFG_O": "Effective Field Goal Percentage",
                   "EFG_D": "Effective Field Goal Percentage Defense",
                   "TOR": "Turnover Rate",
                   "TORD": "Turnover Rate Defense",
                   "ORB": "Offensive Rebound Rate",
                   "DRB": "Defensive Rebound Rate",
                   "FTR": "Free Throw Rate",
                   "FTRD": "Free Throw Rate Defense",
                   "2P_O": "2 Point Percentage",
                   "2P_D": "2 Point Percentage Defense",
                   "3P_O": "3 Point Percentage",
                   "3P_D": "3 Point Percentage Defense",
                   "ADJ_T": "Adjusted Tempo",
                   "WAB": "Wins Above Bubble",
                   }, inplace=True)
df.columns

Index(['TEAM', 'CONF', 'Games Played', 'Wins', 'Adjusted Offensive Efficiency',
       'Adjusted Defensive Efficiency', 'Bartag Score',
       'Effective Field Goal Percentage',
       'Effective Field Goal Percentage Defense', 'Turnover Rate',
       'Turnover Rate Defense', 'Offensive Rebound Rate',
       'Defensive Rebound Rate', 'Free Throw Rate', 'Free Throw Rate Defense',
       '2 Point Percentage', '2 Point Percentage Defense',
       '3 Point Percentage', '3 Point Percentage Defense', 'Adjusted Tempo',
       'Wins Above Bubble', 'POSTSEASON', 'SEED', 'YEAR', 'Post_Season_Score',
       'Seed_Power'],
      dtype='object')

In [6]:
##Sanity Check: Let's see the champions of each season. 
season_champions = df[df["POSTSEASON"] == "Champions"].sort_values("YEAR")

In [ ]:
## Let's see the correlation of various inputs to Post_Season_Score...

inputs = df[["Adjusted Offensive Efficiency", "Adjusted Defensive Efficiency", "Bartag Score",
        "Effective Field Goal Percentage", "Effective Field Goal Percentage Defense",
        "Turnover Rate", "Turnover Rate Defense", "Offensive Rebound Rate", "Defensive Rebound Rate", 
        "Free Throw Rate", "Free Throw Rate Defense",
        "2 Point Percentage", "2 Point Percentage Defense", "3 Point Percentage", "3 Point Percentage Defense",
        "Adjusted Tempo", "Wins Above Bubble", "Games Played", "Wins", "Seed_Power","Post_Season_Score"]].corr()["Post_Season_Score"].sort_values(ascending=False)

inputs


Post_Season_Score                          1.000000
Wins Above Bubble                          0.604975
Seed_Power                                 0.573911
Wins                                       0.563527
Adjusted Offensive Efficiency              0.544232
Bartag Score                               0.533577
Games Played                               0.371823
Effective Field Goal Percentage            0.330576
2 Point Percentage                         0.303341
3 Point Percentage                         0.242643
Offensive Rebound Rate                     0.234867
Turnover Rate Defense                      0.069009
Free Throw Rate                            0.049975
Adjusted Tempo                            -0.035756
Defensive Rebound Rate                    -0.138606
Free Throw Rate Defense                   -0.179942
3 Point Percentage Defense                -0.253593
Turnover Rate                             -0.261825
2 Point Percentage Defense                -0.323406
Effective Fi

In [ ]:
## Keep anything that is correlated above 0.3 or below -0.3. Why? Because ape brain like simple.
reduced_df = df[[
    "Post_Season_Score",
    "Wins Above Bubble",
    "Seed_Power",
    "Wins",
    "Adjusted Offensive Efficiency",
    "Bartag Score",
    "Games Played",
    "Effective Field Goal Percentage",
    "2 Point Percentage",
    "2 Point Percentage Defense",
    "Effective Field Goal Percentage Defense",
    "Adjusted Defensive Efficiency"
]]

reduced_df.corr()["Post_Season_Score"].sort_values(ascending=False)

Post_Season_Score                          1.000000
Wins Above Bubble                          0.604975
Seed_Power                                 0.573911
Wins                                       0.563527
Adjusted Offensive Efficiency              0.544232
Bartag Score                               0.533577
Games Played                               0.371823
Effective Field Goal Percentage            0.330576
2 Point Percentage                         0.303341
2 Point Percentage Defense                -0.323406
Effective Field Goal Percentage Defense   -0.357856
Adjusted Defensive Efficiency             -0.479776
Name: Post_Season_Score, dtype: float64

In [ ]:
# Keep rows with all required features present. Modeling time! Downloading more ram...
print("Starting data cleaning and modeling process...")
reduced_df_clean = reduced_df.dropna(subset=["Seed_Power"])

X = reduced_df_clean.drop("Post_Season_Score", axis=1)
y = reduced_df_clean["Post_Season_Score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred)}")
print(f"R^2 Score: {r2_score(y_test, y_pred)}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")

test_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Difference": y_test.values - y_pred
})
print(test_results)
print(f"\nTest Data Shape: {test_results.shape}")

Starting data cleaning and modeling process...
Mean Squared Error: 0.9546237093522204
R^2 Score: 0.46723302261075106
Mean Absolute Error: 0.7459379629280274
     Actual  Predicted  Difference
0         1   0.550211    0.449789
1         1   1.168630   -0.168630
2         2   3.028807   -1.028807
3         1   0.981429    0.018571
4         1   1.605245   -0.605245
..      ...        ...         ...
145       3   5.390151   -2.390151
146       2   1.746189    0.253811
147       1   1.028713   -0.028713
148       3   3.084692   -0.084692
149       2   1.001024    0.998976

[150 rows x 3 columns]

Test Data Shape: (150, 3)
